# 02 — Experimentos, tabelas e figuras (Kaggle, CPU)

Input: a competição **e** o Dataset de saída do notebook 01 (características).
Ajuste `FEATURES_INPUT` abaixo para o caminho desse Dataset.
Gera `results/` (tabelas CSV, `RESULTS.md`, figuras). Baixe e versione no GitHub.

In [ ]:
# === Setup (Kaggle) ==========================================================
# Pré-requisitos: aceitar as regras da competição "RSNA STR Pulmonary Embolism Detection"
# e adicioná-la como Input deste notebook (+ Add Input -> Competitions).
import os, sys, subprocess, shutil
REPO_URL = "https://github.com/silvarawenne/ap2-topicos.git"
REPO_BRANCH = "main"                      # troque se estiver usando outra branch
REPO = "/kaggle/working/ap2-topicos"
CODE_DATASET = "/kaggle/input/ap2-topicos-code"   # alternativa sem Internet: repo enviado como Dataset

if not os.path.exists(REPO):
    if os.path.exists(CODE_DATASET):
        shutil.copytree(CODE_DATASET, REPO)
    else:  # exige "Internet on" nas configurações do notebook
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements-kaggle.txt"], check=True)
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")
print("repo:", REPO, "| dados:", os.listdir("/kaggle/input"))

In [ ]:
import glob
# Caminho do Dataset com as características (saída do notebook 01). Procura automaticamente:
cands = [p.rsplit("/slices", 1)[0] for p in glob.glob("/kaggle/input/**/slices", recursive=True)]
FEATURES_INPUT = cands[0] if cands else "features"
os.environ["PE_FEATURES_DIR"] = FEATURES_INPUT
print("características:", FEATURES_INPUT)
# A partição/amostra versionadas no repo devem ser as mesmas usadas na extração:
if os.path.exists(f"{FEATURES_INPUT}/splits"):
    shutil.copytree(f"{FEATURES_INPUT}/splits", "splits", dirs_exist_ok=True)

In [ ]:
!python scripts/03_run_experiments.py

In [ ]:
!python scripts/04_make_figures.py --gallery
from IPython.display import Markdown, Image, display
display(Markdown(open("results/RESULTS.md").read()))
for f in sorted(glob.glob("results/figures/*.png")):
    display(Image(f, width=700))

In [ ]:
!cd /kaggle/working && zip -qr tp1_results.zip ap2-topicos/results ap2-topicos/splits && ls -la tp1_results.zip